# 06 — Network Analysis

This notebook converts similarity and association tables into networks.

It adds the course component on graph theory and network metrics: nodes, edges, centrality, and communities.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/ALPcourse_Biblical_Hebrew_Project/biblical_hebrew_genre_analysis")
NOTEBOOK_DIR = PROJECT_DIR / "notebooks"
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_DIR / "output"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"
DOCS_DIR = PROJECT_DIR / "docs"
POSTER_DIR = PROJECT_DIR / "poster"

for folder in [PROCESSED_DIR, TABLES_DIR, FIGURES_DIR, DOCS_DIR, POSTER_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

os.chdir(NOTEBOOK_DIR)
print("Project directory:", PROJECT_DIR)
print("Current folder:", os.getcwd())


!pip -q install pandas matplotlib networkx

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

## 2. Genre similarity network

In [ ]:
similarity_df = pd.read_csv(TABLES_DIR / "genre_cosine_similarity.csv", index_col=0)
similarity_df

In [ ]:
GENRE_SIMILARITY_THRESHOLD = 0.15

G_genre = nx.Graph()

for genre in similarity_df.index:
    G_genre.add_node(genre, node_type="genre")

for i, source in enumerate(similarity_df.index):
    for j, target in enumerate(similarity_df.columns):
        if i < j:
            weight = float(similarity_df.loc[source, target])
            if weight >= GENRE_SIMILARITY_THRESHOLD:
                G_genre.add_edge(source, target, weight=weight)

print("Genre network nodes:", G_genre.number_of_nodes())
print("Genre network edges:", G_genre.number_of_edges())

genre_edges = nx.to_pandas_edgelist(G_genre)
genre_edges.to_csv(TABLES_DIR / "genre_similarity_network_edges.csv", index=False)

centrality = pd.DataFrame({
    "node": list(G_genre.nodes()),
    "degree": [G_genre.degree(n) for n in G_genre.nodes()],
    "degree_centrality": [nx.degree_centrality(G_genre).get(n, 0) for n in G_genre.nodes()],
    "betweenness_centrality": [nx.betweenness_centrality(G_genre, weight="weight").get(n, 0) for n in G_genre.nodes()],
})

centrality.to_csv(TABLES_DIR / "genre_network_centrality.csv", index=False)
centrality

## 3. Visualize genre similarity network

In [ ]:
plt.figure(figsize=(7, 6))

pos = nx.spring_layout(G_genre, seed=42, weight="weight")
weights = [G_genre[u][v]["weight"] * 5 for u, v in G_genre.edges()]

nx.draw_networkx_nodes(G_genre, pos, node_size=1800)
nx.draw_networkx_labels(G_genre, pos, font_size=10)
nx.draw_networkx_edges(G_genre, pos, width=weights)

edge_labels = {(u, v): f'{d["weight"]:.2f}' for u, v, d in G_genre.edges(data=True)}
nx.draw_networkx_edge_labels(G_genre, pos, edge_labels=edge_labels, font_size=8)

plt.title("Genre similarity network")
plt.axis("off")
plt.tight_layout()

filename = FIGURES_DIR / "genre_similarity_network.png"
plt.savefig(filename, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", filename)

## 4. Word association network from PMI

In [ ]:
pmi_path = TABLES_DIR / "pmi_word_associations.csv"
pmi_df = pd.read_csv(pmi_path)

# Build readable labels. Newer PMI tables already include display columns; this fallback
# keeps the notebook reproducible even if only word1/word2 columns are available.
if {"word1_display", "word2_display"}.issubset(pmi_df.columns):
    display_lookup = {}
    for _, row in pmi_df.iterrows():
        display_lookup[str(row["word1"]).strip()] = str(row["word1_display"])
        display_lookup[str(row["word2"]).strip()] = str(row["word2_display"])
else:
    content_for_gloss = pd.read_csv(PROCESSED_DIR / "biblical_hebrew_content_tokens.csv")
    gloss_lookup = (
        content_for_gloss.dropna(subset=["lex_clean", "gloss"])
        .assign(lex_clean=lambda df: df["lex_clean"].astype(str).str.strip(),
                gloss=lambda df: df["gloss"].astype(str).str.strip())
        .drop_duplicates("lex_clean")
        .set_index("lex_clean")["gloss"]
        .to_dict()
    )
    display_lookup = {lex: f"{gloss} ({lex})" for lex, gloss in gloss_lookup.items()}

def display_label(lex):
    lex = str(lex).strip()
    return display_lookup.get(lex, lex)

# Keep a readable, interpretable subset: top 8 PMI pairs per genre.
TOP_PAIRS_PER_GENRE = 8
word_network_rows = (
    pmi_df
    .sort_values(["genre", "pmi"], ascending=[True, False])
    .groupby("genre")
    .head(TOP_PAIRS_PER_GENRE)
    .reset_index(drop=True)
)

G_words = nx.Graph()

for _, row in word_network_rows.iterrows():
    w1, w2 = row["word1"], row["word2"]
    genre = row["genre"]
    weight = float(row["pmi"])

    G_words.add_node(w1, node_type="lexeme", label=display_label(w1))
    G_words.add_node(w2, node_type="lexeme", label=display_label(w2))

    if G_words.has_edge(w1, w2):
        G_words[w1][w2]["weight"] = max(G_words[w1][w2]["weight"], weight)
        G_words[w1][w2]["genres"] += f";{genre}"
    else:
        G_words.add_edge(w1, w2, weight=weight, genres=genre)

print("Word network nodes:", G_words.number_of_nodes())
print("Word network edges:", G_words.number_of_edges())

word_edges = nx.to_pandas_edgelist(G_words)
word_edges.to_csv(TABLES_DIR / "word_association_network_edges.csv", index=False)

word_centrality = pd.DataFrame({
    "node": list(G_words.nodes()),
    "display_label": [display_label(n) for n in G_words.nodes()],
    "degree": [G_words.degree(n) for n in G_words.nodes()],
    "degree_centrality": [nx.degree_centrality(G_words).get(n, 0) for n in G_words.nodes()],
    "betweenness_centrality": [nx.betweenness_centrality(G_words, weight="weight").get(n, 0) for n in G_words.nodes()],
})

try:
    eigen = nx.eigenvector_centrality(G_words, max_iter=1000, weight="weight")
except nx.PowerIterationFailedConvergence:
    eigen = {n: np.nan for n in G_words.nodes()}

word_centrality["eigenvector_centrality"] = word_centrality["node"].map(eigen)
word_centrality = word_centrality.sort_values("degree_centrality", ascending=False)
word_centrality.to_csv(TABLES_DIR / "word_network_centrality.csv", index=False)

word_centrality.head(20)


## 5. Visualize word association network

In [ ]:
# Remove old, less readable non-friendly word-network figures from earlier drafts.
for old_network in FIGURES_DIR.glob("word_association_network*.png"):
    if old_network.name != "word_association_network_friendly.png":
        old_network.unlink()

plt.figure(figsize=(13, 10))

pos = nx.spring_layout(G_words, seed=42, k=0.9, weight="weight")
node_sizes = [350 + 100 * G_words.degree(n) for n in G_words.nodes()]
edge_widths = [max(0.5, G_words[u][v]["weight"] / 2) for u, v in G_words.edges()]
labels = {n: display_label(n) for n in G_words.nodes()}

nx.draw_networkx_nodes(G_words, pos, node_size=node_sizes, alpha=0.85)
nx.draw_networkx_edges(G_words, pos, width=edge_widths, alpha=0.35)
nx.draw_networkx_labels(G_words, pos, labels=labels, font_size=7)

plt.title("Readable PMI-based word association network
Top 8 lexical pairs per genre")
plt.axis("off")
plt.tight_layout()

filename = FIGURES_DIR / "word_association_network_friendly.png"
plt.savefig(filename, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", filename)


## 6. Network interpretation

The genre network represents similarity between genre-level lexical profiles. The word network represents strong PMI-based lexical associations.

Central nodes may be interpreted as lexical hubs, while high-betweenness nodes may function as bridges between clusters of associated words. Because the project is a pilot study, these metrics are exploratory and should be interpreted together with close reading and philological judgment.